#### 로컬 Ollama로 설치한 deepseek-r1 과 Qwen2.5 모델 사용하기
##### ollama pull deepseek-r1:1.5b
##### ollama pull qwen2.5:1.5b
* 소형 모델(0.5B~1.5B)은 9.9 vs 9.11 같은 소수점 비교에서 자주 틀립니다. 정확도가 필요하면 `qwen2.5:3b` 이상으로 `QWEN_MODEL` 상수를 바꿔 사용하세요.

In [ ]:
# 노트북 전체에서 공통으로 사용하는 설정 상수
OLLAMA_BASE_URL = "http://127.0.0.1:11434"   # 로컬 Ollama 서버 주소
DEEPSEEK_MODEL = "deepseek-r1:1.5b"          # 추론(reasoning) 담당 모델
QWEN_MODEL = "qwen2.5:1.5b"                  # 한국어 답변 생성 모델 (설치된 모델 기준)

##### 최신버전 LangChain에서는 ChatOllama와 RunnableSequence(prompt | llm) 를 사용

##### deepseeek 모델 9.9 와 9.11 크기 비교문제  (영어로 질문, invoke()함수)

In [ ]:
import requests

# Ollama 서버 동작 여부 확인 (timeout 지정으로 서버 미실행 시 무한 대기 방지)
try:
    response = requests.get(OLLAMA_BASE_URL, timeout=5)
    print(response.text)
except requests.RequestException as e:
    print(f"Ollama 서버에 연결할 수 없습니다: {e}")

In [ ]:
from langchain_ollama import ChatOllama

try:
    deepseek = ChatOllama(model=DEEPSEEK_MODEL, temperature=0.5, base_url=OLLAMA_BASE_URL)    # 모델 호출
    response = deepseek.invoke("which is bigger between 9.9 and 9.11?")
    print(response.content)
except Exception as e:
    print(f"Error: {e}")

##### qwen2.5:0.5b 모델 9.9 와 9.11 크기 비교문제  (한글로 질문, invoke()함수)

In [ ]:
from langchain_ollama import ChatOllama

try:
    qwen = ChatOllama(model=QWEN_MODEL, temperature=0.5, base_url=OLLAMA_BASE_URL)
    # 모델 호출 (소형 모델이라 오답 가능성이 높음 - 아래 추론 결합 예제에서 개선)
    response = qwen.invoke("9.9와 9.11 중 무엇이 더 큰가요?")
    print(response.content)
except Exception as e:
    print(f"Error: {e}")

##### deepseeek 모델 9.9 와 9.11 크기 비교문제  (영어로 질문, stream()함수)

In [ ]:
from IPython.display import display, Markdown
from langchain_ollama import ChatOllama

deepseek = ChatOllama(model=DEEPSEEK_MODEL, temperature=0.5, base_url=OLLAMA_BASE_URL)

answer = []
for chunk in deepseek.stream("which is bigger between 9.9 and 9.11?"):
    answer.append(chunk.content)
    print(chunk.content, end="", flush=True)


In [ ]:

answer_md=''.join([i for i in answer])
display(Markdown(answer_md))

##### deepseeek 모델 9.9 와 9.11 크기 비교문제  (한글로 질문, stream()함수)

In [ ]:
answer = []
for chunk in deepseek.stream("9.9와 9.11 중 무엇이 더 큰가요?"):
    answer.append(chunk.content)
    print(chunk.content, end="", flush=True)

answer_md=''.join([i for i in answer])
display(Markdown(answer_md))

##### qwen2.5:0.5b 모델 9.9 와 9.11 크기 비교문제  (한글로 질문, stream()함수)

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model=QWEN_MODEL, temperature=0.5, base_url=OLLAMA_BASE_URL)

answer = []
for chunk in model.stream("9.9와 9.11 중 무엇이 더 큰가요?"):
    answer.append(chunk.content)
    print(chunk.content, end="", flush=True)

answer_md=''.join([i for i in answer])
display(Markdown(answer_md))


#### DeepSeek의 추론 능력과 Qwen2의 한글 생성 능력 결합하기
* DeepSeek는 `<think>` 안에서 이루어지는 추론을 기반으로 다른 LLM 대비 높은 성능을 발휘합니다.
* 하지만 Ollama에서 제공하는 deepseek r1 모델은 한국어 생성 능력이 부족합니다.
* DeepSeek의 추론 능력과 Qwen2의 한글 생성 능력 결합해 보겠습니다.
* **주의**: 최신 Ollama는 추론(thinking)을 `content`가 아닌 별도 필드로 분리합니다. 따라서 `stop=["</think>"]`를 쓰면 `content`가 빈 문자열이 되어 추론이 전달되지 않습니다. `reasoning=True`로 설정하고 `additional_kwargs["reasoning_content"]`에서 추론을 꺼내야 합니다.

In [ ]:
# reasoning=True: 추론을 additional_kwargs["reasoning_content"]로 분리해서 받음 (stop 사용 금지)
reasoning_model = ChatOllama(model=DEEPSEEK_MODEL, temperature=0, reasoning=True, base_url=OLLAMA_BASE_URL)
print(reasoning_model)

In [ ]:
generation_model = ChatOllama(model=QWEN_MODEL, temperature=0, base_url=OLLAMA_BASE_URL)
print(generation_model)

#### LangGraph 로 연결하기

In [ ]:
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate

#LangGraph에서 State 사용자정의 클래스는 노드 간의 정보를 전달하는 틀입니다. 
#노드 간에 계속 전달하고 싶거나, 그래프 내에서 유지해야 할 정보를 미리 정의합니다. 
class State(TypedDict):
    question: str
    thinking: str
    answer: str

# 추론(영어)의 결론을 근거로 답하도록 유도 - 소형 모델이 추론을 무시하고 추측하는 것을 방지
answer_prompt = ChatPromptTemplate([
    (
        "system",
        """
        당신은 사용자의 질문에 대해 명확하고 정확한 답변을 제공하는 한국어 AI 어시스턴트입니다.

        당신의 작업은 다음과 같습니다:
        - 제공된 추론(영어)의 최종 결론을 확인하세요.
        - 반드시 추론의 결론과 일치하는 답변을 한국어로 작성하세요.
        - 추론에 없는 내용을 추측해서 덧붙이지 마세요.
        - 답변이 사용자의 질문에 직접적으로 대응하도록 간결하게 작성하세요.

        지침:
        - 대화 형식의 전문적인 톤을 유지하세요.
        - 추론 과정을 사용했다는 사실은 언급하지 마세요.
        """
    ),
    (
        "human",
        """
        질문: {question}
        추론: {thinking}

        위 추론의 결론에 따라 한국어로 답변하세요:
        """
    )
])
print(answer_prompt)

In [ ]:
def extract_thinking(response) -> str:
    """ChatOllama 응답에서 추론 텍스트를 추출한다. 추론이 비어 있으면 ValueError를 발생시킨다."""
    thinking = response.additional_kwargs.get("reasoning_content") or ""
    if not thinking.strip():
        # 일부 버전은 content에 <think>...</think>를 포함해 반환하므로 대체 경로로 처리
        content = response.content or ""
        thinking = content.split("</think>")[0].replace("<think>", "").strip()
    if not thinking.strip():
        raise ValueError("DeepSeek 추론 결과가 비어 있습니다. reasoning=True 설정과 Ollama 버전을 확인하세요.")
    return thinking.strip()

#DeepSeek를 통해서 추론 부분을 생성합니다. Node
def think(state: State):
    response = reasoning_model.invoke(state["question"])
    return {"thinking": extract_thinking(response)}

#Qwen를 통해서 결과 출력 부분을 생성합니다. Node
def generate(state: State):
    messages = answer_prompt.invoke({"question": state["question"], "thinking": state["thinking"]})
    response = generation_model.invoke(messages)
    print(response.content)
    return {"answer": response.content}

# 그래프 컴파일
graph_builder = StateGraph(State).add_sequence([think, generate])
graph_builder.add_edge(START, "think")
graph = graph_builder.compile()
print(type(graph_builder))
print(type(graph))

In [ ]:

# 입력 데이터
inputs = {"question": "9.9와 9.11 중 무엇이 더 큰가요?"}

# invoke()를 사용하여 그래프 호출
result = graph.invoke(inputs)
print(result)

# 결과 출력
print("==> 생성된 답변: \n")
print(result["answer"])

* https://mermaid.live/ 에서  mermain_code 로 직접 확인한다.


In [ ]:
mermaid_code = graph.get_graph().draw_mermaid()
print(mermaid_code)

### 결과에 한글 깨짐 현상이 있는 경우에 강제로 한글로 인코딩 하는 예제

In [ ]:
import os
import sys
from langgraph.graph import START, StateGraph

# UTF-8 인코딩 강제 설정 (Jupyter 노트북 호환)
os.environ['PYTHONIOENCODING'] = 'utf-8'

# Jupyter 환경에서는 reconfigure 대신 환경변수로 처리
try:
    if hasattr(sys.stdout, 'reconfigure') and sys.stdout.encoding != 'utf-8':
        sys.stdout.reconfigure(encoding='utf-8')
except (AttributeError, OSError):
    # Jupyter 노트북이나 다른 환경에서는 패스
    pass

# 앞 셀에서 정의한 State, reasoning_model, generation_model, answer_prompt, extract_thinking 재사용

def ensure_utf8_string(text):
    """문자열이 UTF-8로 제대로 인코딩되었는지 확인하고 변환"""
    if text is None:
        return ""
    if isinstance(text, bytes):
        return text.decode('utf-8', errors='ignore')
    return str(text)

# DeepSeek를 통해서 추론 부분을 생성
def think_debug(state: State):
    question = state["question"]
    print(f"[DEBUG] 입력 질문: {question}")

    response = reasoning_model.invoke(question)
    thinking_content = ensure_utf8_string(extract_thinking(response))

    print(f"[DEBUG] 추론 결과 길이: {len(thinking_content)}")
    print(f"[DEBUG] 추론 결과 미리보기: {thinking_content[:200]}...")

    return {"thinking": thinking_content}

# qwen2.5를 통해서 결과 출력 부분을 생성
def generate_debug(state: State):
    question = ensure_utf8_string(state["question"])
    thinking = ensure_utf8_string(state["thinking"])

    messages = answer_prompt.invoke({"question": question, "thinking": thinking})
    response = generation_model.invoke(messages)
    answer_content = ensure_utf8_string(response.content)

    print(f"[DEBUG] 최종 응답 내용: {answer_content}")

    return {"answer": answer_content}

# 그래프 구성 및 컴파일
debug_graph_builder = StateGraph(State).add_sequence([think_debug, generate_debug])
debug_graph_builder.add_edge(START, "think_debug")
debug_graph = debug_graph_builder.compile()

def main():
    inputs = {"question": "9.9와 9.11 중 무엇이 더 큰가요?"}

    print("="*50)
    print("LangGraph 실행 시작")
    print("="*50)

    try:
        result = debug_graph.invoke(inputs)

        print("="*50)
        print("실행 결과")
        print("="*50)
        print(f"최종 답변: {result.get('answer', '답변 없음')}")

    except Exception as e:
        print(f"오류 발생: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    main()

In [ ]:
inputs = {"question": "9.9와 9.11 중 무엇이 더 큰가요?"}

async for event in graph.astream_events(inputs, version="v2"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        print(event['data']['chunk'].content, end="", flush=True)

### Gradio 를 사용한 챗봇 예제

In [ ]:
import gradio as gr
from langchain_ollama import ChatOllama
from langgraph.graph import START, StateGraph
from typing_extensions import TypedDict
from langchain_core.prompts import ChatPromptTemplate

GRADIO_PORT = 7861   # Gradio 서버 포트

# 모델 설정: 두 개의 서로 다른 모델을 사용하여 추론과 답변 생성을 수행
# - reasoning_model: 추론을 담당하는 모델 (reasoning=True로 추론을 별도 필드로 수신)
# - generation_model: 한국어 답변 생성을 담당하는 모델 (온도 0으로 일관된 답변)
chat_reasoning_model = ChatOllama(model=DEEPSEEK_MODEL, temperature=0, reasoning=True, base_url=OLLAMA_BASE_URL)
chat_generation_model = ChatOllama(model=QWEN_MODEL, temperature=0, base_url=OLLAMA_BASE_URL)

# 상태(State) 정의: 그래프에서 상태를 유지하기 위한 데이터 구조
class ChatState(TypedDict):
    question: str   # 사용자의 질문
    thinking: str   # 추론 결과
    answer: str     # 최종 답변

# 추론의 결론을 근거로 한국어 답변을 생성하는 프롬프트
chat_answer_prompt = ChatPromptTemplate([
    (
        "system",
        """당신은 한국어로만 응답하는 AI 어시스턴트입니다.

        당신의 작업:
        - 제공된 추론(영어)의 최종 결론을 확인하세요.
        - 반드시 추론의 결론과 일치하는 답변을 한국어로 작성하세요.
        - 추론에 없는 내용을 추측해서 덧붙이지 마세요.
        - 질문에 직접적으로 대응하도록 간결하게 작성하세요.
        - 추론 과정을 사용했다는 사실은 언급하지 마세요."""
    ),
    (
        "human",
        """질문: {question}

        추론 과정: {thinking}

        위 추론의 결론에 따라 한국어로 답변하세요:"""
    )
])

def chat_think(state: ChatState):
    """DeepSeek로 추론을 생성한다. (extract_thinking은 앞 셀에서 정의)"""
    response = chat_reasoning_model.invoke(state["question"])
    return {"thinking": extract_thinking(response)}

def chat_generate(state: ChatState):
    """추론 결과를 바탕으로 Qwen이 한국어 최종 답변을 생성한다."""
    messages = chat_answer_prompt.invoke({"question": state["question"], "thinking": state["thinking"]})
    response = chat_generation_model.invoke(messages)
    return {"answer": response.content}

# 그래프는 한 번만 컴파일하여 요청마다 재사용
chat_graph_builder = StateGraph(ChatState).add_sequence([chat_think, chat_generate])
chat_graph_builder.add_edge(START, "chat_think")
chat_graph = chat_graph_builder.compile()

def chatbot_interface(message, history):
    """Gradio ChatInterface 콜백. 오류 시 사용자에게 안내 메시지를 반환한다."""
    try:
        result = chat_graph.invoke({"question": message})
        return result["answer"]
    except Exception as e:
        return f"답변 생성 중 오류가 발생했습니다: {e}"

# Gradio 인터페이스 설정
def launch_gradio():
    iface = gr.ChatInterface(fn=chatbot_interface, title="AI 챗봇", description="질문을 입력하면 AI가 답변을 제공합니다.")
    iface.launch(server_name="127.0.0.1", server_port=GRADIO_PORT)

if __name__ == "__main__":
    launch_gradio()